# Qwen Voice Studio

A small text-to-speech studio powered by **Qwen3-TTS**. Choose a speaker and language, set a delivery direction, then generate, listen to, and download a WAV file. The model runs inside this Kaggle notebook session; it is not an external speech API.

> **First run:** In Kaggle notebook settings, enable **Internet** and select a **GPU** accelerator. The first model load downloads several GB of weights and may take a few minutes.

This notebook uses Qwen's 1.7B CustomVoice model from Hugging Face. Model page: [Qwen3-TTS-12Hz-1.7B-CustomVoice](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice).

In [ ]:
%pip -q install -U qwen-tts gradio

In [ ]:
import gc
import tempfile
from pathlib import Path

import gradio as gr
import soundfile as sf
import torch
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle GPU is required. Open Notebook settings → Accelerator → GPU, then rerun this cell.')

MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Loading Qwen3-TTS from Hugging Face… the first run downloads the model weights.')
tts = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map='cuda:0',
    dtype=torch.float16,
    attn_implementation='sdpa',
)
print('Model ready.')
print('Speakers:', ', '.join(tts.get_supported_speakers()))
print('Languages:', ', '.join(tts.get_supported_languages()))

In [ ]:
SPEAKERS = ['Vivian', 'Serena', 'Uncle_Fu', 'Dylan', 'Eric', 'Ryan', 'Aiden', 'Ono_Anna', 'Sohee']
LANGUAGES = ['Auto', 'English', 'Chinese', 'Japanese', 'Korean', 'German', 'French', 'Russian', 'Portuguese', 'Spanish', 'Italian']

def make_speech(text, speaker, language, delivery):
    text = (text or '').strip()
    if not text:
        raise gr.Error('Add some text before generating speech.')
    if len(text) > 1800:
        raise gr.Error('Please keep each generation under 1,800 characters for a smoother Kaggle session.')
    try:
        with torch.inference_mode():
            wavs, sample_rate = tts.generate_custom_voice(
                text=text,
                language=language,
                speaker=speaker,
                instruct=(delivery.strip() or None),
            )
        out = Path(tempfile.gettempdir()) / 'qwen_voice_studio.wav'
        sf.write(out, wavs[0], sample_rate)
        return str(out), str(out)
    except gr.Error:
        raise
    except Exception as exc:
        raise gr.Error(f'Generation failed: {exc}') from exc

theme = gr.themes.Soft(primary_hue='violet', secondary_hue='slate', neutral_hue='slate')
css = '''
#studio {max-width: 1120px; margin: 0 auto;}
.hero {padding: 24px 4px 12px;}
.hero h1 {font-size: 2.4rem; letter-spacing: -0.04em; margin-bottom: 6px;}
.hero p {font-size: 1rem; opacity: .78;}
.generate button {min-height: 52px; font-size: 1rem; font-weight: 700;}
'''

with gr.Blocks(theme=theme, css=css, title='Qwen Voice Studio') as demo:
    with gr.Column(elem_id='studio'):
        gr.HTML('<div class="hero"><h1>Qwen Voice Studio</h1><p>Turn a script into expressive speech, right here in your Kaggle GPU session.</p></div>')
        with gr.Row(equal_height=True):
            with gr.Column(scale=6):
                text = gr.Textbox(label='Your script', placeholder='Paste or write something to say…', lines=9, max_lines=18, value='Welcome to Qwen Voice Studio. Write a line, choose a voice, and bring your words to life.')
                count = gr.Markdown('**0** characters')
                text.input(lambda value: f'**{len(value or [])}** characters', text, count)
                with gr.Row():
                    speaker = gr.Dropdown(SPEAKERS, value='Ryan', label='Voice', scale=1)
                    language = gr.Dropdown(LANGUAGES, value='English', label='Language', scale=1)
                delivery = gr.Textbox(label='Delivery direction (optional)', placeholder='e.g. Warm and reassuring, with a gentle pace', lines=2)
                generate = gr.Button('Generate speech', variant='primary', elem_classes=['generate'])
            with gr.Column(scale=5):
                gr.Markdown('### Your audio\nGeneration can take a little while, especially the first time.')
                audio = gr.Audio(label='Preview', type='filepath', interactive=False)
                download = gr.File(label='Download WAV', interactive=False)
                gr.Markdown('**Tip:** For best results, match the speaker’s native language. The direction box can suggest mood or pacing.')
        generate.click(make_speech, inputs=[text, speaker, language, delivery], outputs=[audio, download])

demo.queue(default_concurrency_limit=1).launch(
    inline=True,
    share=False,
    prevent_thread_lock=True,
    show_error=True,
)

### Voices

Qwen's built-in speakers include Ryan and Aiden (English), Vivian, Serena, Uncle Fu, Dylan, and Eric (Chinese), Ono Anna (Japanese), and Sohee (Korean). All support the model's listed languages, though the native language generally sounds best.

### If the notebook runs out of GPU memory

Change `1.7B-CustomVoice` to `0.6B-CustomVoice` in `MODEL_ID`, then rerun the model-loading cell and the UI cell. The smaller model uses less memory.